# Hybrid Recommendation System

## Goal

Combine collaborative filtering and content-based retrieval into a single recommendation engine.

## Motivation

Each recommendation approach has strengths and weaknesses.

Collaborative filtering captures user behavior but struggles with cold-start products.

Content-based retrieval handles cold-start products but lacks personalization.

Popularity recommendations provide a reliable fallback for new users.

The hybrid recommender combines these signals to generate robust recommendations.

In [1]:
import pandas as pd
import numpy as np

import faiss
import pickle

In [ ]:
catalog = pd.read_parquet(
    "../data/catalog.parquet"
)

embeddings = np.load(
    "../data/product_embeddings.npy"
)

index = faiss.read_index(
    "../data/faiss_index.bin"
)


# Load Interaction Data

Goal

Reconstruct user-item mappings and sparse interaction matrix required by the ALS model.

These artifacts are recreated from the training data to keep the notebook self-contained and reproducible.

In [3]:
from scipy.sparse import csr_matrix

train_cf = pd.read_parquet(
    "../data/train_cf.parquet"
)

user_ids = train_cf["reviewerID"].unique()

item_ids = train_cf["asin"].unique()

user_to_idx = {
    user: idx
    for idx, user in enumerate(user_ids)
}

item_to_idx = {
    item: idx
    for idx, item in enumerate(item_ids)
}


idx_to_item = {
    v: k
    for k, v in item_to_idx.items()
}

rows = train_cf["reviewerID"].map(
    user_to_idx
)

cols = train_cf["asin"].map(
    item_to_idx
)

data = np.ones(
    len(train_cf)
)


interaction_matrix = csr_matrix(
    (
        data,
        (rows, cols)
    ),
    shape=(
        len(user_ids),
        len(item_ids)
    )
)

print(interaction_matrix.shape)



(190963, 62707)


In [4]:
import pickle
with open(
    "../data/als_model.pkl",
    "rb"
) as f:

    model = pickle.load(f)



user_history = (
    train_cf.groupby("reviewerID")["asin"]
    .apply(set)
    .to_dict()
)

In [5]:
def als_candidates(
    user_id,
    n=50
):

    if user_id not in user_to_idx:
        return []

    user_idx = user_to_idx[user_id]

    ids, scores = model.recommend(
        user_idx,
        interaction_matrix[user_idx],
        N=n
    )

    results = []

    for item_idx, score in zip(
        ids,
        scores
    ):

        asin = idx_to_item[
            item_idx
        ]

        results.append(
            (
                asin,
                float(score)
            )
        )

    return results

In [6]:
asin_to_idx = {
    asin:i
    for i, asin in enumerate(
        catalog["asin"]
    )
}

def content_neighbors(
    asin,
    n=10
):

    if asin not in asin_to_idx:
        return []

    idx = asin_to_idx[asin]

    query = embeddings[
        idx
    ].reshape(1,-1)

    scores, indices = index.search(
        query,
        n + 1
    )

    neighbors = []

    for i in indices[0][1:]:

        neighbors.append(
            catalog.iloc[i]["asin"]
        )

    return neighbors

# Hybrid Ranking

## Goal

Combine multiple recommendation signals into a single ranking score.

## Motivation

Candidate generation identifies potentially relevant products, but does not determine which products should be shown first.

The ranking stage combines:

1. Collaborative filtering signal (ALS)
2. Content-based relevance signal
3. Popularity prior

to produce the final recommendation score.

## Hybrid Score

Hybrid Score =
0.6 × ALS Score
+
0.3 × Content Score
+
0.1 × Popularity Score

In [ ]:
# Build Popularity Lookup
item_popularity = (
    train_cf
    .groupby("asin")
    .size()
    .to_dict()
)

# Normalize Popularity
max_popularity = max(
    item_popularity.values()
)


popularity_top10 = [
    asin for asin, count in sorted(
        item_popularity.items(), 
        key=lambda x: x[1], 
        reverse=True
    )[:10]
]

def popularity_score(
    asin
):
    return (
        item_popularity.get(
            asin,
            0
        )
        /
        max_popularity
    )

# Content Similarity Function

In [8]:
def content_neighbors(
    asin,
    n=10
):

    if asin not in asin_to_idx:
        return []

    idx = asin_to_idx[
        asin
    ]

    query = embeddings[
        idx
    ].reshape(
        1,
        -1
    )

    scores, indices = index.search(
        query,
        n + 1
    )

    results = []

    for sim, i in zip(
        scores[0][1:],
        indices[0][1:]
    ):

        results.append(
            (
                catalog.iloc[i]["asin"],
                float(sim)
            )
        )

    return results

# Hybrid Recommender

In [ ]:
def hybrid_recommend(
    user_id,
    top_k=10
):

    
    # Cold Start

    if user_id not in user_history:

        return catalog[
            catalog["asin"]
            .isin(popularity_top10)
        ][
            ["asin", "title"]
        ]

    
    # ALS Candidates

    raw_als_results = als_candidates(
        user_id,
        n=20
    )

    als_results = []
    
    # Min-Max Scale the ALS scores so they sit exactly between 0.0 and 1.0
    if raw_als_results:
        als_scores = [score for _, score in raw_als_results]
        min_als, max_als = min(als_scores), max(als_scores)
        
        for asin, score in raw_als_results:
            if max_als > min_als:
                scaled_score = (score - min_als) / (max_als - min_als)
            else:
                scaled_score = 1.0 
            
            als_results.append((asin, scaled_score))

    candidate_scores = {}

    
    # ALS Score

    for asin, score in als_results:

        candidate_scores[
            asin
        ] = {
            "als": score,
            "content": 0,
            "popularity": popularity_score(
                asin
            )
        }

    
    # Content Expansion
   

    for asin, _ in als_results:

        neighbors = content_neighbors(
            asin,
            n=5
        )

        for neighbor_asin, sim in neighbors:

            if (
                neighbor_asin
                not in candidate_scores
            ):

                candidate_scores[
                    neighbor_asin
                ] = {
                    "als": 0,
                    "content": sim,
                    "popularity": popularity_score(
                        neighbor_asin
                    )
                }

            else:

                candidate_scores[
                    neighbor_asin
                ]["content"] = max(
                    candidate_scores[
                        neighbor_asin
                    ]["content"],
                    sim
                )

    
    # Remove Seen Items

    seen = user_history[
        user_id
    ]

    for item in list(
        candidate_scores.keys()
    ):

        if item in seen:

            del candidate_scores[
                item
            ]

    
    # Hybrid Score

    ranked = []

    for asin, scores in (
        candidate_scores.items()
    ):

        final_score = (
            0.6 * scores["als"]
            +
            0.3 * scores["content"]
            +
            0.1 * scores["popularity"]
        )

        ranked.append(
            (
                asin,
                final_score
            )
        )

    ranked.sort(
        key=lambda x: x[1],
        reverse=True
    )

    final_asins = [
        asin
        for asin, _
        in ranked[:top_k]
    ]

    # return catalog[
    #     catalog["asin"]
    #     .isin(final_asins)
    # ][
    #     [
    #         "asin",
    #         "title"
    #     ]
    # ]

    results = []

    for asin, score in ranked[:top_k]:

        title = catalog.loc[
            catalog["asin"] == asin,
            "title"
        ]

        if len(title):

            results.append(
                {
                    "asin": asin,
                    "title": title.iloc[0],
                    "hybrid_score": score
                }
            )

    return pd.DataFrame(results)

    

# Hybrid Recommendation Example

Generate recommendations using collaborative filtering, semantic retrieval, and popularity priors.

In [14]:
sample_user = (
    train_cf["reviewerID"]
    .iloc[500]
)

hybrid_recommend(
    sample_user,
    top_k=10
)

,asin,title,hybrid_score
0,B0088CJT4U,TP-LINK TL-WDR4300 Wireless N750 Dual Band Rou...,0.615247
1,B00829TIEK,Seagate Backup Plus 3TB USB 3.0 Desktop Extern...,0.601411
2,B000N99BBC,TP-LINK TL-SG1005D 10/100/1000Mbps 5-Port Giga...,0.558449
3,B004CLYEDC,"Micra Digital CAT5e Snagless Patch Cable, 5 Fe...",0.463699
4,B00829THK0,Seagate Backup Plus 1TB Desktop External Hard ...,0.458605
5,B004CLYEFK,Micra Digital USB A to USB B Cable (6 Feet),0.443582
6,B004CLYEE6,Micra Digital CAT6 Snagless Patch Cable; 5 Fee...,0.306140
7,B00HFRWWAM,Seagate Backup Plus 3TB Desktop External Hard ...,0.300590
8,B00BOHNYTW,Seagate Backup Plus Slim 500GB Portable Hard D...,0.299396
9,B00829THH8,Seagate Backup Plus 1TB Portable External Hard...,0.294035


In [11]:
def explain_recommendation(
    user_id,
    asin
):
    print(f"Recommended ASIN: {asin}\n")
    print("Signals Used:")

    # 1. Check if user triggered the Cold-Start Fallback
    if user_id not in user_history:
        print("- Popularity prior (Cold-Start Fallback)")
        return
        
    # 2. Check if the item originated from the ALS model
    als_recs = [item for item, score in als_candidates(user_id, n=20)]
    
    if asin in als_recs:
        print("- Collaborative filtering (ALS)")
    else:
        print("- Semantic similarity (Content Expansion)")
        
    # 3. Every candidate in the hybrid equation gets a popularity adjustment
    print("- Popularity prior")

In [12]:
sample_recs = hybrid_recommend(
    sample_user,
    top_k=10
)

asin = sample_recs.iloc[
    0
]["asin"]

explain_recommendation(
    sample_user,
    asin
)

Recommended ASIN: B0015AARJI

Signals Used:
- Collaborative filtering (ALS)
- Popularity prior
